# 04 — Multilingual transformer fine-tuning (human / ai / fusion)

One notebook for every multilingual backbone. Pick the model with `HW3_MODEL` (environment variable) or by editing
`MODEL_KEY`; outputs are tagged with the key so runs never overwrite each other.

| Key | Backbone | Why it is in the plan |
|---|---|---|
| `mdeberta` | `microsoft/mdeberta-v3-base` (~280M) | strongest small fine-tuned detector in multilingual MGT benchmarks (MULTITuDE, MultiSocial) |
| `xlmr` | `FacebookAI/xlm-roberta-base` (~280M) | the classic multilingual baseline; reference for `xlmr_detect` |
| `xlmr_detect` | `almanach/xlmr-chatgptdetect-noisy` | XLM-R-base already fine-tuned for ChatGPT detection (HC3, en+fr, MIT). `xlmr` vs `xlmr_detect` isolates *does detector pre-training help?* |
| `mmbert` | `jhu-clsp/mmBERT-base` (~307M, 8k context) | different architecture and tokenizer (Gemma 2), reads 1024 tokens: diversity for the ensemble and a long-text comparison |

All runs: fixed training split (`splits/train_ids.csv`), fixed validation split, macro-F1, head+tail truncation for
texts longer than `max_len`, class-weighted cross-entropy, linear warm-up/decay, fp16 autocast, best epoch kept.
Outputs: `outputs/val_scores/04_<key>.csv`, `outputs/test_scores/04_<key>.csv`, `submissions/04_<key>.csv`, a row in
`outputs/results.csv`.

**RX 6800 XT (ROCm):** select the `../.venv` kernel. One-time install there (torch is already the ROCm build):
`uv pip install --python ../.venv/bin/python transformers sentencepiece` (or `../.venv/bin/python -m pip install …`).
Headless queue example, from the repo root:
`HW3_MODEL=mdeberta nohup ../.venv/bin/jupyter nbconvert --to notebook --execute --output 04_mdeberta.ipynb --ExecutePreprocessor.timeout=-1 src/experiments/04_multilingual_finetune.ipynb > outputs/04_mdeberta.log 2>&1 &`

Smoke test: `HW3_TRAIN_N=500 HW3_EPOCHS=1` in front of the same command. Running headless (not inside VS Code) keeps
the editor's memory out of the way and the run survives an editor crash; follow it with `tail -f outputs/04_<key>.log`.

In [ ]:
import os
import sys
from pathlib import Path

ON_KAGGLE = Path("/kaggle/input").exists()
if ON_KAGGLE:
    inp = Path("/kaggle/input")
    os.environ["HW3_DATA_DIR"] = str(next(inp.rglob("train.csv")).parent)
    os.environ["HW3_SPLITS_DIR"] = str(next(inp.rglob("val_ids.csv")).parent)
    os.environ["HW3_OUTPUT_DIR"] = "/kaggle/working/outputs"
    os.environ["HW3_SUBMISSION_DIR"] = "/kaggle/working/submissions"
    SRC = next(inp.rglob("config.py")).parent
else:
    SRC = next(q for p in [Path.cwd(), *Path.cwd().parents] for q in (p, p / "src") if (q / "config.py").exists())
sys.path.insert(0, str(SRC))

In [ ]:
import gc
import math
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer

from config import LABELS, OUTPUT_DIR, SEED
from utils import load_splits, load_test, log_result, macro_f1, save_submission, save_val_scores

MODELS = {
    "mdeberta":    dict(id="microsoft/mdeberta-v3-base",        max_len=512,  head=128, lr=2e-5, batch=8, accum=2),
    "xlmr":        dict(id="FacebookAI/xlm-roberta-base",       max_len=512,  head=128, lr=2e-5, batch=8, accum=2),
    "xlmr_detect": dict(id="almanach/xlmr-chatgptdetect-noisy", max_len=512,  head=128, lr=2e-5, batch=8, accum=2),
    "mmbert":      dict(id="jhu-clsp/mmBERT-base",              max_len=1024, head=256, lr=3e-5, batch=4, accum=4),
}
MODEL_KEY = os.environ.get("HW3_MODEL", "mdeberta")
M = MODELS[MODEL_KEY]
MAX_LEN, HEAD_TOKENS = M["max_len"], M["head"]   # long texts keep the first HEAD_TOKENS and the last ones
BATCH, ACCUM = M["batch"], M["accum"]             # effective batch 16; if out of GPU memory halve BATCH, double ACCUM
LR_BACKBONE, LR_HEAD = M["lr"], 1e-4
TRAIN_N = int(os.environ.get("HW3_TRAIN_N", 0)) or None   # None = whole training split; smoke test: HW3_TRAIN_N=500
EPOCHS = int(os.environ.get("HW3_EPOCHS", 2))
WARMUP = 0.1              # share of optimizer steps with linear warm-up
WEIGHT_DECAY = 0.01
CLASS_WEIGHTS = True      # inverse-frequency loss weights (macro-F1 counts every class equally)
GRAD_CHECKPOINTING = True # less GPU memory, ~30% slower
ZERO_SHOT_N = 2_000       # validation texts for a zero-shot check when the checkpoint is already a detector
EVAL_BATCH = 32
LOG_EVERY = 100           # optimizer steps between progress lines

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"    # ROCm builds also report "cuda"
FP16 = DEVICE == "cuda"   # if the loss turns NaN (seen with DeBERTa-v3 in fp16), set FP16 = False
TAG = f"04_{MODEL_KEY}"
CKPT = OUTPUT_DIR / f"{TAG}_best.pt"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
torch.manual_seed(SEED); np.random.seed(SEED)
print(f"model={MODEL_KEY} ({M['id']})  device={DEVICE} {torch.cuda.get_device_name(0) if DEVICE == 'cuda' else ''}  fp16={FP16}")

## Tokenisation and batching

In [ ]:
tok = AutoTokenizer.from_pretrained(M["id"])
tok.model_max_length = 10**9          # we truncate ourselves; silences the "sequence too long" warning

# special tokens around a text, read from the tokenizer itself (works for [CLS]/[SEP], <s>/</s>, <bos>/<eos>)
_raw = tok("x", add_special_tokens=False)["input_ids"]
_full = tok("x")["input_ids"]
_i = next(i for i in range(len(_full)) if _full[i:i + len(_raw)] == _raw)
PREFIX, SUFFIX = _full[:_i], _full[_i + len(_raw):]


def encode(texts: pd.Series, chunk: int = 2_000) -> list:
    """Token ids with head+tail truncation to MAX_LEN, as int32 arrays.

    Tokenized in chunks: tokenizing all texts at once keeps every full-length token list in RAM (~100 bytes per
    token, several GB here). Texts are cut at 30k chars first to bound tokenizer time.
    """
    body = MAX_LEN - len(PREFIX) - len(SUFFIX)
    tail = body - HEAD_TOKENS
    texts = texts.astype(str).str.slice(0, 30_000)
    out = []
    for i in range(0, len(texts), chunk):
        for x in tok(texts.iloc[i:i + chunk].tolist(), add_special_tokens=False)["input_ids"]:
            out.append(np.array(PREFIX + (x if len(x) <= body else x[:HEAD_TOKENS] + x[-tail:]) + SUFFIX, dtype=np.int32))
    return out


def batches(seqs, labels=None, bs=8, rng=None):
    """Yield (row indices, input_ids, attention_mask, labels); shuffled if rng is given, else sorted by length."""
    order = rng.permutation(len(seqs)) if rng is not None else np.argsort([len(s) for s in seqs])
    pad = tok.pad_token_id if tok.pad_token_id is not None else 0
    for i in range(0, len(order), bs):
        idx = order[i:i + bs]
        L = max(len(seqs[j]) for j in idx)
        ids = torch.full((len(idx), L), pad, dtype=torch.long)
        att = torch.zeros((len(idx), L), dtype=torch.long)
        for r, j in enumerate(idx):
            ids[r, :len(seqs[j])] = torch.from_numpy(seqs[j])
            att[r, :len(seqs[j])] = 1
        yield idx, ids, att, (torch.as_tensor(labels[idx]) if labels is not None else None)


@torch.no_grad()
def predict_logits(model, seqs) -> np.ndarray:
    model.eval()
    out = np.zeros((len(seqs), model.config.num_labels), dtype=np.float32)
    for idx, ids, att, _ in batches(seqs, bs=EVAL_BATCH):
        with torch.autocast("cuda", dtype=torch.float16, enabled=FP16):
            out[idx] = model(input_ids=ids.to(DEVICE), attention_mask=att.to(DEVICE)).logits.float().cpu().numpy()
    return out

In [ ]:
train_df, val_df = load_splits()
test_df = load_test()
if TRAIN_N and TRAIN_N < len(train_df):
    train_df = train_test_split(train_df, train_size=TRAIN_N, stratify=train_df.domain + "|" + train_df.label,
                                random_state=SEED)[0].reset_index(drop=True)
lab2id = {l: i for i, l in enumerate(LABELS)}
y_tr = train_df.label.map(lab2id).to_numpy()
y_va = val_df.label.to_numpy()

t0 = time.time()
tr_seqs, va_seqs = encode(train_df.text), encode(val_df.text)
lens = np.array([len(s) for s in tr_seqs])
print(f"train={len(train_df):,}  val={len(val_df):,}  tokenized in {time.time() - t0:.0f}s; "
      f"{(lens >= MAX_LEN).mean():.0%} of training texts hit MAX_LEN={MAX_LEN} (head+tail truncated)")
for lang in sorted(train_df.lang.unique()):
    m = (train_df.lang == lang).to_numpy()
    print(f"  {lang}: median {np.median(lens[m]):.0f} tokens, {(lens[m] >= MAX_LEN).mean():.0%} truncated")

## 1. Zero-shot check (only for checkpoints that are already detectors)

In [ ]:
cfg = AutoConfig.from_pretrained(M["id"])
ai_like = [i for i, name in cfg.id2label.items() if any(k in str(name).lower() for k in ("ai", "gpt", "machine", "generated", "fake"))]
if ZERO_SHOT_N and getattr(cfg, "architectures", None) and "SequenceClassification" in cfg.architectures[0] and len(ai_like) == 1:
    zs_model = AutoModelForSequenceClassification.from_pretrained(M["id"]).to(DEVICE)
    zs = val_df.sample(min(ZERO_SHOT_N, len(val_df)), random_state=SEED).reset_index(drop=True)
    zs["p_ai"] = torch.softmax(torch.tensor(predict_logits(zs_model, encode(zs.text))), 1)[:, ai_like[0]].numpy()
    rows = []
    for name, d in [("all", zs)] + [(l, g) for l, g in zs.groupby("lang")]:
        auc = lambda pos, neg: (roc_auc_score(d.label[d.label.isin([pos, neg])] == pos, d.p_ai[d.label.isin([pos, neg])])
                                if d.label.isin([pos]).any() and d.label.isin([neg]).any() else np.nan)
        rows.append({"subset": name, "AUC ai vs human": auc("ai", "human"), "AUC fusion vs human": auc("fusion", "human"),
                     "AUC ai vs fusion": auc("ai", "fusion")})
    print(f"zero-shot, AI label = {cfg.id2label[ai_like[0]]!r}")
    print(pd.DataFrame(rows).round(3).to_string(index=False))
    del zs_model; gc.collect(); torch.cuda.empty_cache() if DEVICE == "cuda" else None
else:
    print("not a detector checkpoint (or zero-shot disabled) - skipped")

## 2. Fine-tuning

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    M["id"], num_labels=len(LABELS), id2label=dict(enumerate(LABELS)), label2id=lab2id,
    ignore_mismatched_sizes=True,          # replaces a detector's 2-class output layer by a new 3-class one
).float().to(DEVICE)                       # fp32 master weights; fp16 only inside autocast (transformers v5 may load fp16)
if GRAD_CHECKPOINTING:
    try:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    except Exception as e:                 # some architectures do not support it
        print("gradient checkpointing not available:", e)

prefix = model.base_model_prefix + "."
no_decay = lambda n: n.endswith("bias") or "norm" in n.lower()
named = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
backbone = [(n, p) for n, p in named if n.startswith(prefix)]
head = [p for n, p in named if not n.startswith(prefix)]
opt = torch.optim.AdamW([
    {"params": [p for n, p in backbone if not no_decay(n)], "lr": LR_BACKBONE, "weight_decay": WEIGHT_DECAY},
    {"params": [p for n, p in backbone if no_decay(n)], "lr": LR_BACKBONE, "weight_decay": 0.0},
    {"params": head, "lr": LR_HEAD, "weight_decay": 0.0},
])
total_steps = EPOCHS * math.ceil(len(tr_seqs) / (BATCH * ACCUM))
warm = max(1, int(WARMUP * total_steps))
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda s: min((s + 1) / warm, max(0.0, (total_steps - s) / max(1, total_steps - warm))))
scaler = torch.amp.GradScaler("cuda", enabled=FP16)

counts = np.bincount(y_tr, minlength=len(LABELS))
weights = torch.tensor(counts.sum() / (len(LABELS) * counts), dtype=torch.float32) if CLASS_WEIGHTS else None
loss_fn = nn.CrossEntropyLoss(weight=weights.to(DEVICE) if weights is not None else None)
print(f"parameters: backbone {sum(p.numel() for _, p in backbone) / 1e6:.0f}M, head {sum(p.numel() for p in head) / 1e3:.0f}k; "
      f"optimizer steps {total_steps}, warm-up {warm}")

In [ ]:
rng = np.random.default_rng(SEED)
best_f1, step, history = -1.0, 0, []
t_start = time.time()
for epoch in range(EPOCHS):
    model.train()
    running, n_seen = 0.0, 0
    opt.zero_grad(set_to_none=True)
    for b, (idx, ids, att, yb) in enumerate(batches(tr_seqs, y_tr, bs=BATCH, rng=rng)):
        with torch.autocast("cuda", dtype=torch.float16, enabled=FP16):
            logits = model(input_ids=ids.to(DEVICE), attention_mask=att.to(DEVICE)).logits
        loss = loss_fn(logits.float(), yb.to(DEVICE)) / ACCUM
        if not torch.isfinite(loss):
            raise RuntimeError("loss is NaN/inf - set FP16 = False or lower the learning rate")
        scaler.scale(loss).backward()
        running += loss.item() * ACCUM; n_seen += 1
        if (b + 1) % ACCUM == 0:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            opt.zero_grad(set_to_none=True)
            step += 1
            if step % LOG_EVERY == 0:
                el = time.time() - t_start
                print(f"epoch {epoch + 1} step {step}/{total_steps}  loss {running / n_seen:.4f}  "
                      f"elapsed {el / 60:.0f} min  ETA {el / step * (total_steps - step) / 60:.0f} min", flush=True)
                running, n_seen = 0.0, 0

    va_logits = predict_logits(model, va_seqs)
    pred = np.array(LABELS)[va_logits.argmax(1)]
    f1 = macro_f1(y_va, pred)
    history.append({"epoch": epoch + 1, "val_macro_f1": f1, "minutes": (time.time() - t_start) / 60})
    print(f"== epoch {epoch + 1}: validation macro-F1 = {f1:.4f}", flush=True)
    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), CKPT)
        np.save(OUTPUT_DIR / f"{TAG}_val_logits.npy", va_logits)
print(pd.DataFrame(history).round(4).to_string(index=False))

## 3. Evaluation of the best epoch (fixed validation split)

In [ ]:
va_prob = torch.softmax(torch.tensor(np.load(OUTPUT_DIR / f"{TAG}_val_logits.npy")), 1).numpy()
pred_va = np.array(LABELS)[va_prob.argmax(1)]
print(f"{MODEL_KEY}: macro-F1 = {macro_f1(y_va, pred_va):.4f}\n")
print(classification_report(y_va, pred_va, labels=LABELS, digits=3))
cm = confusion_matrix(y_va, pred_va, labels=LABELS)
print("Confusion matrix (rows = true, cols = predicted):")
print(pd.DataFrame(cm, index=LABELS, columns=LABELS).to_string())
h, f = lab2id["human"], lab2id["fusion"]
print(f"human<->fusion confusion rate: {(cm[h, f] + cm[f, h]) / (cm[h].sum() + cm[f].sum()):.3f}")

ev = val_df[["domain", "lang"]].assign(y=y_va, pred=pred_va)
rows = [{"group": col, "value": v, "n": len(g), "macro_f1": macro_f1(g.y, g.pred)}
        for col in ("lang", "domain") for v, g in ev.groupby(col)]
print("\n" + pd.DataFrame(rows).round(3).to_string(index=False))

log_result("04_encoder", f"{MODEL_KEY} | n={len(train_df)} ep={EPOCHS} len={MAX_LEN} head+tail", y_va, pred_va,
           notes=M["id"])
save_val_scores(TAG, val_df.id, va_prob, LABELS)

## 4. Test predictions

In [ ]:
model.load_state_dict(torch.load(CKPT, map_location=DEVICE))
te_prob = torch.softmax(torch.tensor(predict_logits(model, encode(test_df.text))), 1).numpy()
(OUTPUT_DIR / "test_scores").mkdir(exist_ok=True)
pd.DataFrame(te_prob, columns=LABELS).assign(id=test_df.id.values).to_csv(OUTPUT_DIR / "test_scores" / f"{TAG}.csv", index=False)
save_submission(test_df.id, np.array(LABELS)[te_prob.argmax(1)], TAG)

res = pd.read_csv(OUTPUT_DIR / "results.csv")
print(res.loc[res.groupby("model").macro_f1.idxmax()].sort_values("macro_f1", ascending=False)
      [["experiment", "model", "macro_f1", "f1_human", "f1_ai", "f1_fusion"]].head(10).to_string(index=False))